In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import geopandas as gpd


In [ ]:
df = pd.read_csv('Heracleum mantegazzianum.csv', low_memory=False)
df.sample(10)

In [ ]:
df.info(verbose=True)
df.describe()

From the initial data description, it is visible that not all columns have the correct data types. The *eventDate* column should be formatted in **datetime** instead of in **string**, the *total_observations* column is formatted as **float**, and while that's usable, it should be of the **integer** type instead, and finally the *Heracleum mantegazzianum* column is in **string**, while that should be an **integer** column as well.

In [ ]:
df[df.duplicated()]

From the initial data description, it is visible that not all columns have the correct data types. The *eventDate* column should be formatted in **datetime** instead of in **string**, the *total_observations* column is formatted as **float**, and while that's usable, it should be of the **integer** type instead, and finally the *Heracleum mantegazzianum* column is in **string**, while that should be an **integer** column as well.

From the above duplicate check, it is also visible that there are **no** duplicates. (Ruben)

In [ ]:
for col in df.columns:
    print(f"\n{col}:\n", df[col].unique())
print(df.head(5))

TLDR;
some **NAN** values in total_observations
Heracleum mantegazzianum array has some *unknown* and *-1 negative values* (Mika)

In [ ]:
df['total_observations'] = pd.to_numeric(df['total_observations'], errors='coerce')
df['total_observations'] = df['total_observations'].fillna(0).astype(int)
#print(df['total_observations'].dtype, len(df['total_observations']))

I observed that the total_observations column was with float values and NaNs. To mitigate this, I converted all NaNs to 0 and all values from float to int. (Marcell)

In [ ]:
df['Heracleum mantegazzianum'] = pd.to_numeric(df['Heracleum mantegazzianum'], errors='coerce')
df['Heracleum mantegazzianum'] = df['Heracleum mantegazzianum'].fillna(0).astype(int)
df['Heracleum mantegazzianum'] = df['Heracleum mantegazzianum'].clip(lower=0)
#print(df['Heracleum mantegazzianum'].isnull().sum(), df['Heracleum mantegazzianum'].dtype, len(df['Heracleum mantegazzianum']))

The Heracleum mantegazzianum column was originally in a string format. All unknown / NaN values were converted to 0 and all others including the new 0s were converted to integers. Furthermore, all values below 0 were raised to 0 as there was at least 1 instance where the row value was below 0 (-1). (Marcell)
I observed that the total_observations column had a; only round numbers and b; potential NaN values.
To mitigate this, I first converted every entry into either a number or an NaN value. Then, I filled every NaN with a 0 and converted all floats to integers. Because of domain knowledge and performance reasons, namely that an observation has to be a round number (There's no such thing as 0.5 observation), I converted all floats to integers.

In [ ]:
nl = gpd.read_file("gadm41_NLD.gpkg", layer="ADM_ADM_1")

fig, ax = plt.subplots(figsize=(15, 10))
nl.plot(ax=ax, color='lightgray', edgecolor='white')
sns.histplot(
    data=df,
    x=df['decimalLongitude'],
    y=df['decimalLatitude'],
    weights=df['total_observations'],
    cbar=True,
    cbar_kws={
        'label' : 'Total observations'
    },
    binwidth=(0.05, 0.05),
    ax=ax,
    alpha=0.6,
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Distribution of total observations and their coördinate locations")

plt.show()

I took the latitude and longitude coordinates from the observations, and applied them in a *distplot* to show ***distribution of observations*** by using the *total_observations* column as a weight. This is done to learn where most observations are made. This means we have also learnt that some observations appear to have been made in ***Somalia***. Either that or the coordinates of those observations have been **inverted**. This is something to fix before we move forward.

We can either choose to *invert* these values and fix them, or choose to *remove them. Ideally, we shall do the former.